In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Reproducibility
np.random.seed(42)
random.seed(42)

print("Environment Ready ✅")


Environment Ready ✅


In [2]:
# --------- BASIC CONFIG ---------
N_CUSTOMERS = 20000

start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 12, 31)

date_range_days = (end_date - start_date).days

# --------- GENERATE CUSTOMER IDS ---------
customer_ids = [f"CUST_{i:05d}" for i in range(1, N_CUSTOMERS + 1)]

# --------- GENERATE SIGNUP DATES ---------
signup_dates = [
    start_date + timedelta(days=np.random.randint(0, date_range_days))
    for _ in range(N_CUSTOMERS)
]

# --------- REGION DISTRIBUTION ---------
regions = ["South India", "North India", "West India", "International"]
region_probs = [0.4, 0.3, 0.2, 0.1]

assigned_regions = np.random.choice(regions, N_CUSTOMERS, p=region_probs)

# --------- COMPANY SIZE ---------
company_sizes = ["Small", "Medium", "Large"]
size_probs = [0.5, 0.35, 0.15]

assigned_sizes = np.random.choice(company_sizes, N_CUSTOMERS, p=size_probs)

# --------- INDUSTRY ---------
industries = ["Tech", "Retail", "Finance", "Healthcare"]
assigned_industries = np.random.choice(industries, N_CUSTOMERS)

# --------- ACQUISITION CHANNEL ---------
channels = ["Organic", "Paid Ads", "Referral"]
channel_probs = [0.4, 0.4, 0.2]

assigned_channels = np.random.choice(channels, N_CUSTOMERS, p=channel_probs)

# --------- CREATE DATAFRAME ---------
customers_df = pd.DataFrame({
    "customer_id": customer_ids,
    "signup_date": signup_dates,
    "region": assigned_regions,
    "company_size": assigned_sizes,
    "industry": assigned_industries,
    "acquisition_channel": assigned_channels
})

customers_df.head()


,customer_id,signup_date,region,company_size,industry,acquisition_channel
0,CUST_00001,2024-04-12,North India,Small,Healthcare,Organic
1,CUST_00002,2025-03-11,North India,Small,Finance,Organic
2,CUST_00003,2024-09-27,South India,Medium,Tech,Referral
3,CUST_00004,2024-04-16,International,Medium,Healthcare,Organic
4,CUST_00005,2024-03-12,South India,Large,Finance,Paid Ads


In [3]:
# --------- PLAN CONFIGURATION ---------
plans = ["Basic", "Pro", "Enterprise"]
plan_probs = [0.5, 0.35, 0.15]

plan_prices = {
    "Basic": 999,
    "Pro": 2499,
    "Enterprise": 6999
}

# --------- ASSIGN PLANS ---------
assigned_plans = np.random.choice(plans, N_CUSTOMERS, p=plan_probs)

monthly_prices = [plan_prices[p] for p in assigned_plans]

# --------- CREATE SUBSCRIPTION TABLE ---------
subscriptions_df = pd.DataFrame({
    "customer_id": customers_df["customer_id"],
    "plan": assigned_plans,
    "monthly_price": monthly_prices,
    "start_date": customers_df["signup_date"],
    "upgrade_flag": 0  # initially no upgrades
})

subscriptions_df.head()


,customer_id,plan,monthly_price,start_date,upgrade_flag
0,CUST_00001,Basic,999,2024-04-12,0
1,CUST_00002,Basic,999,2025-03-11,0
2,CUST_00003,Basic,999,2024-09-27,0
3,CUST_00004,Basic,999,2024-04-16,0
4,CUST_00005,Pro,2499,2024-03-12,0


In [4]:
subscriptions_df["plan"].value_counts(normalize=True) * 100


plan
Basic         49.70
Pro           35.57
Enterprise    14.73
Name: proportion, dtype: float64

In [5]:
# --------- GENERATE MONTHLY DATE RANGE ---------

monthly_range = pd.date_range(start="2024-01-01", 
                              end="2025-12-01", 
                              freq="MS")  # MS = Month Start

months_df = pd.DataFrame({
    "month": monthly_range
})

months_df.head()


,month
0,2024-01-01
1,2024-02-01
2,2024-03-01
3,2024-04-01
4,2024-05-01


In [6]:
len(months_df)


24

In [7]:
# --------- EXPAND CUSTOMER MONTH DATA ---------

customer_month_list = []

for idx, row in customers_df.iterrows():
    signup_month = row["signup_date"].replace(day=1)
    
    active_months = monthly_range[monthly_range >= signup_month]
    
    for month in active_months:
        customer_month_list.append([row["customer_id"], month])

customer_month_df = pd.DataFrame(customer_month_list, 
                                 columns=["customer_id", "month"])

print("Total Rows Created:", len(customer_month_df))

customer_month_df.head()


Total Rows Created: 249508


,customer_id,month
0,CUST_00001,2024-04-01
1,CUST_00001,2024-05-01
2,CUST_00001,2024-06-01
3,CUST_00001,2024-07-01
4,CUST_00001,2024-08-01


In [8]:
customer_month_df["customer_id"].nunique()


20000

In [9]:
# --------- MERGE CUSTOMER INFO ---------

# Merge subscription data
customer_month_df = customer_month_df.merge(
    subscriptions_df[["customer_id", "plan", "monthly_price"]],
    on="customer_id",
    how="left"
)

# Merge customer profile data
customer_month_df = customer_month_df.merge(
    customers_df[[
        "customer_id", 
        "region", 
        "company_size", 
        "industry", 
        "acquisition_channel"
    ]],
    on="customer_id",
    how="left"
)

customer_month_df.head()


,customer_id,month,plan,monthly_price,region,company_size,industry,acquisition_channel
0,CUST_00001,2024-04-01,Basic,999,North India,Small,Healthcare,Organic
1,CUST_00001,2024-05-01,Basic,999,North India,Small,Healthcare,Organic
2,CUST_00001,2024-06-01,Basic,999,North India,Small,Healthcare,Organic
3,CUST_00001,2024-07-01,Basic,999,North India,Small,Healthcare,Organic
4,CUST_00001,2024-08-01,Basic,999,North India,Small,Healthcare,Organic


In [10]:
# --------- GENERATE USAGE METRICS ---------

def generate_usage(row):
    
    # Base login based on plan
    if row["plan"] == "Basic":
        base_login = np.random.normal(8, 3)
    elif row["plan"] == "Pro":
        base_login = np.random.normal(20, 5)
    else:  # Enterprise
        base_login = np.random.normal(35, 7)
    
    # Company size effect
    if row["company_size"] == "Large":
        base_login *= 1.2
    elif row["company_size"] == "Small":
        base_login *= 0.9
    
    # Seasonality (Q4 boost)
    if row["month"].month in [10, 11, 12]:
        base_login *= 1.15
    
    # Random drop chance (engagement decline simulation)
    if np.random.rand() < 0.05:
        base_login *= 0.4
    
    login_count = max(0, round(base_login))
    
    # Average session time (depends on plan)
    if row["plan"] == "Basic":
        session_time = np.random.normal(15, 5)
    elif row["plan"] == "Pro":
        session_time = np.random.normal(30, 8)
    else:
        session_time = np.random.normal(50, 10)
    
    session_time = max(5, round(session_time, 2))
    
    # Feature usage score (0–100 scaled by logins)
    feature_usage = min(100, max(0, login_count * np.random.uniform(2, 4)))
    
    return pd.Series([login_count, session_time, feature_usage])


customer_month_df[["login_count", "avg_session_time", "feature_usage_score"]] = (
    customer_month_df.apply(generate_usage, axis=1)
)

customer_month_df.head()


,customer_id,month,plan,monthly_price,region,company_size,industry,acquisition_channel,login_count,avg_session_time,feature_usage_score
0,CUST_00001,2024-04-01,Basic,999,North India,Small,Healthcare,Organic,3.0,11.29,10.508498
1,CUST_00001,2024-05-01,Basic,999,North India,Small,Healthcare,Organic,6.0,19.20,22.112784
2,CUST_00001,2024-06-01,Basic,999,North India,Small,Healthcare,Organic,6.0,18.94,19.939134
3,CUST_00001,2024-07-01,Basic,999,North India,Small,Healthcare,Organic,9.0,15.43,33.623250
4,CUST_00001,2024-08-01,Basic,999,North India,Small,Healthcare,Organic,7.0,11.93,22.317276


In [11]:
customer_month_df["login_count"].mean()

16.179942126104173

In [12]:
# --------- GENERATE PAYMENT DATA ---------

def generate_payment(row):
    
    plan_price = row["monthly_price"]
    
    # Missed payment chance
    missed_payment_prob = 0.06
    
    if np.random.rand() < missed_payment_prob:
        return pd.Series([0, np.random.randint(30, 60)])
    
    # Normal payment
    amount_paid = plan_price
    
    # Delay behavior by plan
    if row["plan"] == "Basic":
        delay = np.random.choice([0, 1, 2, 5, 10], p=[0.6, 0.15, 0.1, 0.1, 0.05])
    elif row["plan"] == "Pro":
        delay = np.random.choice([0, 1, 2, 3], p=[0.75, 0.15, 0.07, 0.03])
    else:  # Enterprise
        delay = np.random.choice([0, 1], p=[0.9, 0.1])
    
    return pd.Series([amount_paid, delay])


customer_month_df[["amount_paid", "payment_delay_days"]] = (
    customer_month_df.apply(generate_payment, axis=1)
)

customer_month_df.head()


,customer_id,month,plan,monthly_price,region,company_size,industry,acquisition_channel,login_count,avg_session_time,feature_usage_score,amount_paid,payment_delay_days
0,CUST_00001,2024-04-01,Basic,999,North India,Small,Healthcare,Organic,3.0,11.29,10.508498,999,0
1,CUST_00001,2024-05-01,Basic,999,North India,Small,Healthcare,Organic,6.0,19.20,22.112784,999,0
2,CUST_00001,2024-06-01,Basic,999,North India,Small,Healthcare,Organic,6.0,18.94,19.939134,999,0
3,CUST_00001,2024-07-01,Basic,999,North India,Small,Healthcare,Organic,9.0,15.43,33.623250,999,0
4,CUST_00001,2024-08-01,Basic,999,North India,Small,Healthcare,Organic,7.0,11.93,22.317276,999,0


In [13]:
customer_month_df["payment_delay_days"].describe()


count    249508.000000
mean          3.447328
std          10.748815
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          59.000000
Name: payment_delay_days, dtype: float64

In [14]:
(customer_month_df["amount_paid"] == 0).mean() * 100


5.996200522628533

In [15]:
# --------- GENERATE SUPPORT TICKETS ---------

def generate_tickets(row):
    
    # Base rate by plan
    if row["plan"] == "Basic":
        lam = 0.6
    elif row["plan"] == "Pro":
        lam = 0.4
    else:
        lam = 0.3
    
    # If payment delay occurred → frustration
    if row["payment_delay_days"] > 5:
        lam += 0.5
    
    # Occasional service outage simulation
    if np.random.rand() < 0.02:
        lam += 2
    
    ticket_count = np.random.poisson(lam)
    
    return ticket_count


customer_month_df["ticket_count"] = customer_month_df.apply(generate_tickets, axis=1)

customer_month_df.head()


,customer_id,month,plan,monthly_price,region,company_size,industry,acquisition_channel,login_count,avg_session_time,feature_usage_score,amount_paid,payment_delay_days,ticket_count
0,CUST_00001,2024-04-01,Basic,999,North India,Small,Healthcare,Organic,3.0,11.29,10.508498,999,0,1
1,CUST_00001,2024-05-01,Basic,999,North India,Small,Healthcare,Organic,6.0,19.20,22.112784,999,0,0
2,CUST_00001,2024-06-01,Basic,999,North India,Small,Healthcare,Organic,6.0,18.94,19.939134,999,0,1
3,CUST_00001,2024-07-01,Basic,999,North India,Small,Healthcare,Organic,9.0,15.43,33.623250,999,0,1
4,CUST_00001,2024-08-01,Basic,999,North India,Small,Healthcare,Organic,7.0,11.93,22.317276,999,0,0


In [16]:
customer_month_df["ticket_count"].mean()


0.5669277137406415

In [17]:
(customer_month_df["ticket_count"] > 3).mean() * 100


0.8620966061208457

In [25]:
customers_df = customers_df.drop(columns=["churn"], errors="ignore")


In [26]:
# --------- AGGREGATE CUSTOMER BEHAVIOR ---------

customer_behavior = customer_month_df.groupby("customer_id").agg({
    "login_count": "mean",
    "feature_usage_score": "mean",
    "ticket_count": "mean",
    "payment_delay_days": "mean",
    "month": "count"
}).reset_index()

customer_behavior.rename(columns={"month": "tenure_months"}, inplace=True)

# Merge plan and region back
customer_behavior = customer_behavior.merge(
    subscriptions_df[["customer_id", "plan"]], 
    on="customer_id", 
    how="left"
)

customer_behavior = customer_behavior.merge(
    customers_df[["customer_id", "region"]], 
    on="customer_id", 
    how="left"
)

# --------- CHURN PROBABILITY MODEL ---------

def assign_churn(row):
    score = 0
    
    # Low engagement
    if row["login_count"] < 10:
        score += 2
        
    if row["feature_usage_score"] < 30:
        score += 2
    
    # High tickets
    if row["ticket_count"] > 1:
        score += 2
        
    # Payment delay
    if row["payment_delay_days"] > 5:
        score += 2
    
    # Plan risk
    if row["plan"] == "Basic":
        score += 1
    
    # Region risk
    if row["region"] == "North India":
        score += 1
    
    # Early churn tendency
    if row["tenure_months"] < 6:
        score += 2
    
    # Convert score to probability
    churn_prob = min(1, score / 18)
    
    return 1 if np.random.rand() < churn_prob else 0


customer_behavior["churn"] = customer_behavior.apply(assign_churn, axis=1)

# Merge churn back to customer table
customers_df = customers_df.merge(
    customer_behavior[["customer_id", "churn"]],
    on="customer_id",
    how="left"
)

customers_df.head()


,customer_id,signup_date,region,company_size,industry,acquisition_channel,churn_x,churn_y,churn
0,CUST_00001,2024-04-12,North India,Small,Healthcare,Organic,1,1,1
1,CUST_00002,2025-03-11,North India,Small,Finance,Organic,1,0,0
2,CUST_00003,2024-09-27,South India,Medium,Tech,Referral,1,1,1
3,CUST_00004,2024-04-16,International,Medium,Healthcare,Organic,1,0,1
4,CUST_00005,2024-03-12,South India,Large,Finance,Paid Ads,0,0,0


In [27]:
customers_df["churn"].mean() * 100


19.615

In [28]:
# Merge churn into monthly data
customer_month_df = customer_month_df.merge(
    customers_df[["customer_id", "churn"]],
    on="customer_id",
    how="left"
)

customer_month_df.head()


,customer_id,month,plan,monthly_price,region,company_size,industry,acquisition_channel,login_count,avg_session_time,feature_usage_score,amount_paid,payment_delay_days,ticket_count,churn
0,CUST_00001,2024-04-01,Basic,999,North India,Small,Healthcare,Organic,3.0,11.29,10.508498,999,0,1,1
1,CUST_00001,2024-05-01,Basic,999,North India,Small,Healthcare,Organic,6.0,19.20,22.112784,999,0,0,1
2,CUST_00001,2024-06-01,Basic,999,North India,Small,Healthcare,Organic,6.0,18.94,19.939134,999,0,1,1
3,CUST_00001,2024-07-01,Basic,999,North India,Small,Healthcare,Organic,9.0,15.43,33.623250,999,0,1,1
4,CUST_00001,2024-08-01,Basic,999,North India,Small,Healthcare,Organic,7.0,11.93,22.317276,999,0,0,1


In [29]:
# --------- MONTHLY REVENUE AGGREGATION ---------

monthly_revenue_df = customer_month_df.groupby("month").agg({
    "amount_paid": "sum",
    "customer_id": "nunique",
    "churn": "sum"
}).reset_index()

monthly_revenue_df.rename(columns={
    "amount_paid": "total_revenue",
    "customer_id": "active_customers",
    "churn": "churn_count"
}, inplace=True)

monthly_revenue_df.head()


,month,total_revenue,active_customers,churn_count
0,2024-01-01,2012190,867,144
1,2024-02-01,3698973,1624,279
2,2024-03-01,5649645,2498,439
3,2024-04-01,7502862,3341,583
4,2024-05-01,9225111,4162,735


In [30]:
monthly_revenue_df.tail()


,month,total_revenue,active_customers,churn_count
19,2025-08-01,37602891,16632,3055
20,2025-09-01,39713046,17455,3292
21,2025-10-01,41710746,18359,3514
22,2025-11-01,43567971,19186,3736
23,2025-12-01,45436215,20000,3923


In [31]:
# --------- SIMULATE MARKETING SPEND ---------

marketing_data = []

base_spend = 500000  # starting monthly spend (₹)

for i, row in monthly_revenue_df.iterrows():
    
    month = row["month"]
    
    # Gradual growth in marketing budget
    growth_factor = 1 + (i * 0.02)
    spend = base_spend * growth_factor
    
    # Q4 Boost
    if month.month in [10, 11, 12]:
        spend *= 1.15
    
    # Economic slowdown simulation (May–July 2024)
    if month >= pd.to_datetime("2024-05-01") and month <= pd.to_datetime("2024-07-01"):
        spend *= 0.85
    
    # Random noise
    spend *= np.random.uniform(0.95, 1.05)
    
    # Estimate new customers
    new_customers = int(spend / 5000)
    
    marketing_data.append([month, int(spend), new_customers])


marketing_spend_df = pd.DataFrame(
    marketing_data,
    columns=["month", "marketing_spend", "new_customers_acquired"]
)

marketing_spend_df.head()


,month,marketing_spend,new_customers_acquired
0,2024-01-01,484196,96
1,2024-02-01,509298,101
2,2024-03-01,514479,102
3,2024-04-01,536100,107
4,2024-05-01,470670,94


In [32]:
marketing_spend_df.tail()


,month,marketing_spend,new_customers_acquired
19,2025-08-01,684986,136
20,2025-09-01,713732,142
21,2025-10-01,808651,161
22,2025-11-01,795797,159
23,2025-12-01,801947,160


In [34]:
import os

# Create folder structure if not exists
os.makedirs("data/raw", exist_ok=True)

print("Folder structure ready ✅")


Folder structure ready ✅


In [35]:
# --------- SAVE RAW DATA ---------

customers_df.to_csv("data/raw/customers.csv", index=False)
subscriptions_df.to_csv("data/raw/subscriptions.csv", index=False)
customer_month_df.to_csv("data/raw/customer_month_data.csv", index=False)
monthly_revenue_df.to_csv("data/raw/monthly_revenue.csv", index=False)
marketing_spend_df.to_csv("data/raw/marketing_spend.csv", index=False)

print("All raw datasets saved successfully ✅")


All raw datasets saved successfully ✅


In [36]:
import os
os.getcwd()


'C:\\Users\\gokul\\OneDrive\\Desktop\\financial_health_twin\\notebooks'